In [1]:
import pandas as pd
import os
import glob

PASTA_FICHEIROS = r"C:\Users\LISARR\Documents\python\01.Financeiro\weekly_input_csi"
COLUNA_ORIGEM = "ficheiro_origem"
CAMPO_DATA = "FENTREGA"

ficheiros_xls = glob.glob(os.path.join(PASTA_FICHEIROS, "*.xls"))
print(f"Encontrados {len(ficheiros_xls)} ficheiros")

lista_dfs = []

for ficheiro in ficheiros_xls:
    try:
        df_temp = pd.read_excel(ficheiro, engine='xlrd')  # Força o engine xlrd
        df_temp[COLUNA_ORIGEM] = os.path.basename(ficheiro)
        lista_dfs.append(df_temp)
        print(f"  ✓ {os.path.basename(ficheiro)}: {len(df_temp)} linhas")
    except Exception as e:
        print(f"  ✗ Erro ao ler {os.path.basename(ficheiro)}: {e}")

if lista_dfs:
    df = pd.concat(lista_dfs, ignore_index=True)
    df[CAMPO_DATA] = pd.to_datetime(df[CAMPO_DATA], errors='coerce')
    print(f"\n✓ DataFrame criado: {df.shape[0]} linhas, {df.shape[1]} colunas")
    df.head()

Encontrados 1 ficheiros
  ✗ Erro ao ler SAL_DAT027 (11).xls: Unsupported format, or corrupt file: Expected BOF record; found b'<?xml ve'


In [2]:
import xml.etree.ElementTree as ET

def ler_linhas_xml_spreadsheet(caminho_ficheiro):
    """
    Lê um ficheiro no formato SpreadsheetML (Excel XML 2003)
    e devolve (cabecalho, linhas)
    """
    tree = ET.parse(caminho_ficheiro)
    root = tree.getroot()
    
    worksheet = root.find('.//{urn:schemas-microsoft-com:office:spreadsheet}Worksheet')
    table = worksheet.find('{urn:schemas-microsoft-com:office:spreadsheet}Table')
    
    rows = table.findall('{urn:schemas-microsoft-com:office:spreadsheet}Row')
    
    todas_linhas = []
    for row in rows:
        cells = row.findall('{urn:schemas-microsoft-com:office:spreadsheet}Cell')
        linha = []
        for cell in cells:
            data = cell.find('{urn:schemas-microsoft-com:office:spreadsheet}Data')
            valor = data.text if data is not None else ""
            linha.append(valor)
        todas_linhas.append(linha)
    
    if not todas_linhas:
        return None, None
    
    cabecalho = todas_linhas[0]
    linhas = todas_linhas[1:]
    
    return cabecalho, linhas

In [ ]:
# ==============================================================================
# LEITURA DE FICHEIROS PARA DATAFRAME
# ==============================================================================

import glob

ficheiros_xls = glob.glob(os.path.join(PASTA_FICHEIROS, "*.xls"))
print(f"Encontrados {len(ficheiros_xls)} ficheiros")

lista_dfs = []

for ficheiro in ficheiros_xls:
    cabecalho, linhas = ler_linhas_xml_spreadsheet(ficheiro)
    
    if cabecalho and linhas:
        df_temp = pd.DataFrame(linhas, columns=cabecalho)
        df_temp[COLUNA_ORIGEM] = os.path.basename(ficheiro)
        lista_dfs.append(df_temp)
        print(f"  ✓ {os.path.basename(ficheiro)}: {len(df_temp)} linhas")

if lista_dfs:
    df = pd.concat(lista_dfs, ignore_index=True)
    df[CAMPO_DATA] = pd.to_datetime(df[CAMPO_DATA], errors='coerce')
    print(f"\n✓ DataFrame criado: {df.shape[0]} linhas, {df.shape[1]} colunas")
    df.head()
else:
    print("Nenhum ficheiro foi carregado")

Encontrados 1 ficheiros


In [ ]:
# ==============================================================================
# FILTROS: PROPIETARIO (CEP, PTG) + MÊS (FENTREGA)
# ==============================================================================

MES_FILTRO = 8  # <-- define aqui o mês desejado (1-12)

df_filtrado = df[
    (df["PROPIETARIO"].isin(["CEP", "PTG"])) &
    (df[CAMPO_DATA].dt.month == MES_FILTRO)
]

print(f"Linhas antes do filtro: {len(df)}")
print(f"Linhas depois do filtro: {len(df_filtrado)}")

df_filtrado.head()

Linhas antes do filtro: 55199
Linhas depois do filtro: 6681


,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,PESO_BRUTO,...,LOCDES,LUGARDESCARGA,TEMP_MERC_PED,TIPOPALETA,CAMION_TIPO,CAMION_CAPACIDAD,TIPO_COMBUSTIBLE,KMREALES,ALBARAN,ficheiro_origem
16146,CEP,SALVESEN LOGISTICA PORTUGAL / AUCHAN RETAIL PO...,SALVESEN,0002HGS,0002HGS,15.79,0,15.79,2.36,1227.17,...,706584,AUCHAN - AZAMBUJA,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,0,NO,SAL_DAT027 (1).xls
16147,CEP,SALVESEN LOGISTICA PORTUGAL / AUCHAN RETAIL PO...,SALVESEN,0002HGS,0002HGS,0.49,0,0.49,0.1,37.44,...,706584,AUCHAN - AZAMBUJA,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,0,NO,SAL_DAT027 (1).xls
16148,CEP,SALVESEN LOGISTICA PORTUGAL / AUCHAN RETAIL PO...,SALVESEN,0002HGS,0002HGS,67.92,0,67.92,8,1338.67,...,746383,AUCHAN - SALVESEN - REFRIGERADO,TAM,EUR,Trailer 33 plts,33,DIESEL,0,NO,SAL_DAT027 (1).xls
16149,PTG,SALVESEN LOGISTICA PORTUGAL / AUCHAN RETAIL PO...,SALVESEN,0002HGS,0002HGS,14.16,0,14.16,2.32,646.5,...,527060,AUCHAN RETAIL PORTUGAL,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,55.48,NO,SAL_DAT027 (1).xls
16150,PTG,SALVESEN LOGISTICA PORTUGAL / AUCHAN RETAIL PO...,SALVESEN,0002HGS,0002HGS,7.3,0,7.3,1,501,...,727278,AUCHAN SALVESEN,@@@,EUR,Trailer 33 plts,33,DIESEL,0,NO,SAL_DAT027 (1).xls


In [ ]:
# ==============================================================================
# RESUMO POR FENTREGA: CUSTO, PALETS, CUSTO/PALETE
# ==============================================================================

df_filtrado["COSTEDT"] = (
    df_filtrado["COSTEDT"].astype(str).str.replace(",", ".").astype(float)
)
df_filtrado["PALETS"] = (
    df_filtrado["PALETS"].astype(str).str.replace(",", ".").astype(float)
)

resumo_dia = df_filtrado.groupby(CAMPO_DATA).agg(
    CUSTO=("COSTEDT", "sum"),
    PALETS=("PALETS", "sum")
).reset_index()

resumo_dia["CUSTO_POR_PALETE"] = resumo_dia["CUSTO"] / resumo_dia["PALETS"]

resumo_dia["CUSTO"] = resumo_dia["CUSTO"].apply(lambda x: f"{x:,.0f}".replace(",", "."))

resumo_dia

,FENTREGA,CUSTO,PALETS,CUSTO_POR_PALETE
0,2026-08-01,10.437,899.0,11.609655
1,2026-08-03,20.919,1812.0,11.544917
2,2026-08-04,22.099,1982.0,11.149823
3,2026-08-05,23.652,2461.0,9.610825
4,2026-08-06,21.808,2226.0,9.797075
5,2026-08-07,24.035,2523.0,9.526243
6,2026-08-08,10.089,966.0,10.444089
7,2026-08-10,20.714,1856.0,11.160722
8,2026-08-11,22.029,2263.0,9.734322
9,2026-08-12,21.484,2469.0,8.701559


In [ ]:
# ==============================================================================
# FILTROS: PROPIETARIO (CEP, PTG) + MÊS + GRAVAR ATÉ AO DIA ESCOLHIDO
# ==============================================================================

MES_FILTRO = 8   # <-- define o mês (1-12)
DIA_FIM = 22     # <-- define o dia final do período

CAMINHO_CSV = r"C:\Users\LISARR\Documents\Infor_27_weekly\SAL_DAT027.csv"

df_filtrado = df[
    (df["PROPIETARIO"].isin(["CEP", "PTG"])) &
    (df[CAMPO_DATA].dt.month == MES_FILTRO)
]

data_fim = df_filtrado[CAMPO_DATA].dt.to_period("M").iloc[0].start_time.replace(day=DIA_FIM)
df_periodo = df_filtrado[df_filtrado[CAMPO_DATA] <= data_fim]

for col in ["INGRESODT", "COSTEDT", "RENTADT"]:
    df_periodo[col] = (
        df_periodo[col].astype(str).str.replace(",", ".").astype(float)
    )

df_periodo.to_csv(CAMINHO_CSV, index=False, sep=";", encoding="utf-8-sig", date_format="%Y%m%d")

print(f"✓ Mês: {MES_FILTRO} | Até dia: {DIA_FIM}")
print(f"✓ Linhas gravadas: {len(df_periodo)}")
print(f"✓ Ficheiro gravado: {CAMINHO_CSV}")

✓ Mês: 8 | Até dia: 11
✓ Linhas gravadas: 5282
✓ Ficheiro gravado: C:\Users\LISARR\Documents\Infor_27_weekly\SAL_DAT027.csv


In [ ]:
# ==============================================================================
# RESUMO POR PRESTADOR (TRANSPORTISTA) - SELECIONAR DIA
# ==============================================================================

DIA_SELECIONADO = 11  # <-- define aqui o dia do mês (ex: 8)

df_dia = df_filtrado[df_filtrado[CAMPO_DATA].dt.day == DIA_SELECIONADO].copy()

df_dia["COSTEDT"] = df_dia["COSTEDT"].astype(str).str.replace(",", ".").astype(float)
df_dia["PALETS"] = df_dia["PALETS"].astype(str).str.replace(",", ".").astype(float)

resumo_prestador = df_dia.groupby("TRANSPORTISTA").agg(
    CUSTO=("COSTEDT", "sum"),
    PALETS=("PALETS", "sum")
).reset_index()

resumo_prestador["CUSTO_POR_PALETE"] = resumo_prestador["CUSTO"] / resumo_prestador["PALETS"]

resumo_prestador["CUSTO"] = resumo_prestador["CUSTO"].apply(lambda x: f"{x:,.0f}".replace(",", "."))

resumo_prestador

,TRANSPORTISTA,CUSTO,PALETS,CUSTO_POR_PALETE
0,CMTIR TRANSPORTES NAC. INTERN. S.A,2.633,155.0,16.985097
1,J. MARCOS FABIO E SEATLLE TRANSPORTES LDA,310,9.0,34.446667
2,JMR PRESTAÇAO DE SERVIÇOS PARA A DISTRIBUÇAO S.A,3.202,458.0,6.991441
3,MODELO CONTINENTE HIPERMERCADOS S.A.,780,99.0,7.877576
4,RAMITRANS TRANSPORTES LDA,727,64.0,11.359687
5,SALVESEN,0,92.0,0.000000
6,TIERS SPOT,0,1.0,0.000000
7,"TJA-TRANSPORTES J.AMARAL, S.A. -",981,90.0,10.898333
8,TORRESTIR - TRANSPORTES NACIONAIS E INTERNACIO...,1.000,47.0,21.276383
9,TRANSAURA TRANSPORTES LDA,468,59.0,7.931525


In [ ]:
# ==============================================================================
# DETALHE POR DIA + TRANSPORTADOR: CODEUT, PALETS, CUSTO, ORIGEM, DESTINO
# ==============================================================================

DIA_SELECIONADO = 10  # <-- define aqui o dia do mês
TRANSPORTADOR_SELECIONADO = "TRANSPORTES FIGUEIREDO & FIGUEIREDO, LDA"  # <-- define aqui o transportador

df_detalhe = df_filtrado[
    (df_filtrado[CAMPO_DATA].dt.day == DIA_SELECIONADO) &
    (df_filtrado["TRANSPORTISTA"] == TRANSPORTADOR_SELECIONADO)
]

detalhe = df_detalhe[["CODEUT", "PALETS", "COSTEDT", "LUGARCARGA", "LUGARDESCARGA"]].copy()
detalhe["COSTEDT"] = detalhe["COSTEDT"].astype(str).str.replace(",", ".").astype(float)
detalhe["COSTEDT"] = detalhe["COSTEDT"].apply(lambda x: f"{x:,.0f}".replace(",", "."))

detalhe

,CODEUT,PALETS,COSTEDT,LUGARCARGA,LUGARDESCARGA
39541,3515746,2,77,AUCHAN CASTELO BRANCO,SALVESEN LOGISTICA PORTUGAL
39542,3515746,1,39,AUCHAN CASTELO BRANCO,SALVESEN LOGISTICA PORTUGAL
39543,3515746,3,116,"CONFEITARIA CARLOS GONÇALVES,",LIDL BELGIUM GMBH UND CO.KG
39544,3515746,1,1,SALVESEN LOGISTICA AZAMBUJA 2,DANIGURTE DIST. PRODUTOS ALIME
39545,3515746,7,249,SALVESEN LOGISTICA PORTUGAL,DANIGURTE
39546,3515746,1,20,SALVESEN LOGISTICA AZAMBUJA 2,DANIGURTE
39547,3515746,1,17,SALVESEN LOGISTICA PORTUGAL,AUCHAN CASTELO BRANCO
39548,3515746,1,3,SALVESEN LOGISTICA PORTUGAL,A.S VILA VELHA RODAO I (11548)
39549,3515746,1,3,SALVESEN LOGISTICA PORTUGAL,A.S VILA VELHA RODAO II (11545)
39550,3515746,1,0,SALVESEN LOGISTICA PORTUGAL,A.S. VILA VELHA RÓDÃO (S/N)
